In [8]:
"""
한국 전체 기업 시계열 예측
범용 버전 - 노트북/데스크톱 어디서나 작동
"""

# ==================== 1. 범용 경로 설정 ====================
import sys
import os
from pathlib import Path

def setup_universal_paths():
    """
    어떤 PC에서도 작동하는 범용 경로 설정
    DATA 폴더를 자동으로 찾아 경로 추가
    """
    current = Path.cwd()

    # 상위 폴더를 탐색하며 DATA 폴더 찾기
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            # 프로젝트 루트와 DATA 폴더 모두 추가
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            if str(data_folder) not in sys.path:
                sys.path.insert(0, str(data_folder))

            print("=" * 70)
            print("📁 경로 설정 완료")
            print("=" * 70)
            print(f"✓ 프로젝트 루트: {parent}")
            print(f"✓ DATA 폴더:    {data_folder}")
            print(f"✓ 현재 위치:     {current}")
            print(f"✓ 운영체제:      {os.name}")
            print("=" * 70 + "\n")

            return {
                'project_root': parent,
                'data_folder': data_folder,
                'current': current
            }

    # 못 찾으면 에러
    raise FileNotFoundError(
        f"❌ DATA 폴더를 찾을 수 없습니다.\n"
        f"현재 위치: {current}\n"
        f"상위 폴더에 DATA 폴더가 있는지 확인하세요."
    )

# 경로 설정 실행
try:
    paths = setup_universal_paths()
except FileNotFoundError as e:
    print(e)
    print("\n대안: 수동으로 경로를 설정하세요.")
    # sys.path.insert(0, "여기에_프로젝트_루트_경로_입력")
    sys.exit(1)


# ==================== 2. 필요한 모듈 import ====================
try:
    # 예측 함수 import (파일명 확인 필요!)
    from universal_ts_forecast_function import (
        forecast_one_from_pivot_inline,
        monitor_memory_usage
    )
    from stock_invest_function import fetch_table_data

    print("✓ 예측 모듈 import 성공")

except ImportError as e:
    print(f"❌ 모듈 import 실패: {e}")
    print("\n확인 사항:")
    print("1. DATA 폴더에 'universal_ts_forecast_function.py' 파일이 있는가?")
    print("2. DATA 폴더에 'stock_invest_function.py' 파일이 있는가?")
    print("\n파일명이 다르다면 위 import 문을 수정하세요.")
    sys.exit(1)

from DATA.stock_invest_function import *

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("✓ 모든 모듈 import 완료\n")


# ==================== 3. 설정 ====================
# 데이터베이스 연결 정보 (본인 환경에 맞게 수정)
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),  # 노트북에서는 다른 IP일 수 있음
    'port': '3307',
    'database': 'investar'
}

# 예측 설정
TARGET_TICKER = "A005930"  # 삼성전자
FORECAST_HORIZON = 12      # 12분기 예측
TARGET_INDICATOR = '순이익률'
MODELS_TO_USE = ["SARIMA", "ETS", "Prophet",  "LSTM"]  # 사용할 모델

print("⚙️  설정 정보")
print(f"  - 대상 티커: {TARGET_TICKER}")
print(f"  - 예측 기간: {FORECAST_HORIZON}분기")
print(f"  - 사용 모델: {', '.join(MODELS_TO_USE)}")
print(f"  - 데이터베이스: {db_info['host']}:{db_info['port']}\n")


# ==================== 4. 데이터 로드 ====================
print("=" * 70)
print("1단계: 데이터 로드")
print("=" * 70)

try:
    fs_df = fetch_table_data(db_info, "korea_fs_data")
    print(f"✓ 데이터 로드 완료: {len(fs_df):,}행")

except Exception as e:
    print(f"❌ 데이터베이스 연결 실패: {e}")
    print("\n확인 사항:")
    print("1. 데이터베이스가 실행 중인가?")
    print("2. 네트워크 연결이 정상인가?")
    print("3. 호스트 주소가 올바른가?")
    print(f"   현재 설정: {db_info['host']}:{db_info['port']}")
    sys.exit(1)


# ==================== 5. 데이터 전처리 ====================
print("\n" + "=" * 70)
print("2단계: 데이터 전처리")
print("=" * 70)

# 날짜 컬럼명 변경
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 특정 지표 필터링
filtered_df = fs_df[fs_df['indicator'] == TARGET_INDICATOR].copy()
print(f"✓ 지표 필터링: {TARGET_INDICATOR} - {len(filtered_df):,}행")

# 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)
filtered_df.rename(columns={'symbol': 'ticker'}, inplace=True)

# value 컬럼 타입 변환
if 'value' not in filtered_df.columns:
    print("❌ 'value' 컬럼이 없습니다.")
    sys.exit(1)

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 피벗 테이블 생성
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='ticker',
    values='value',
    aggfunc='first'
)

print(f"✓ 피벗 테이블 생성: {pivot_df.shape}")

# 날짜 범위 제한
pivot_df.index.name = 'date'
pivot_df = pivot_df.loc['2010-03-31': '2025-06-30']
print(f"✓ 날짜 범위: {pivot_df.index.min()} ~ {pivot_df.index.max()}")
print(f"✓ 티커 수: {len(pivot_df.columns):,}개")

# 메모리 정리
del fs_df, filtered_df
import gc
gc.collect()

# 메모리 모니터링
print("\n메모리 상태 확인:")
monitor_memory_usage(threshold_mb=2000)


# ==================== 6. 예측 실행 ====================
print("\n" + "=" * 70)
print("3단계: 예측 실행")
print("=" * 70)

try:
    result = forecast_one_from_pivot_inline(
        pivot_df=pivot_df,
        target_col=TARGET_TICKER,
        horizon=FORECAST_HORIZON,
        models=MODELS_TO_USE,
        strict_no_nan=True
    )

    print("\n✓ 예측 완료!")

except Exception as e:
    print(f"\n❌ 예측 실행 중 오류: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)


# ==================== 7. 결과 출력 ====================
print("\n" + "=" * 70)
print("4단계: 예측 결과")
print("=" * 70)

for model_name, info in result.items():
    print(f"\n▶ {model_name}")

    if "error" in info:
        print(f"  ✗ 오류: {info['error']}")
    else:
        # 메타 정보
        if "spec" in info:
            print(f"  · 모델 사양: {info['spec']}")

        # 변환 정보
        if "used_transform" in info:
            print(f"  · 사용 변환: {info['used_transform']}")

        # 예측값
        if "forecast" in info and not isinstance(info["forecast"], dict):
            forecast_values = np.array(info["forecast"]).flatten()
            print(f"  · 예측값 (처음 5개): {np.round(forecast_values[:5], 2)}")
            print(f"  · 예측값 범위: {forecast_values.min():.2f} ~ {forecast_values.max():.2f}")


# ==================== 8. DataFrame 생성 ====================
print("\n" + "=" * 70)
print("5단계: 결과 DataFrame 생성")
print("=" * 70)

forecasts = {}

for model_name, info in result.items():
    if "forecast" in info and not isinstance(info["forecast"], dict):
        forecasts[model_name] = np.array(info["forecast"]).flatten()
    else:
        forecasts[model_name] = np.full(FORECAST_HORIZON, np.nan)

# DataFrame 변환
forecast_df = pd.DataFrame(forecasts)

# 예측 구간 자동 추정
from universal_ts_forecast_function import infer_freq_alias

last_date = pivot_df.index.max()
freq_alias = infer_freq_alias(pivot_df.index)

print(f"✓ 마지막 날짜: {last_date}")
print(f"✓ 주기: {freq_alias}")

if freq_alias == "M":
    future_index = pd.date_range(
        last_date + pd.offsets.MonthEnd(1),
        periods=FORECAST_HORIZON,
        freq="M"
    )
elif freq_alias == "Q":
    future_index = pd.date_range(
        last_date + pd.offsets.QuarterEnd(1),
        periods=FORECAST_HORIZON,
        freq="Q"
    )
elif freq_alias == "D":
    future_index = pd.date_range(
        last_date + pd.Timedelta(days=1),
        periods=FORECAST_HORIZON,
        freq="D"
    )
else:
    future_index = pd.date_range(
        last_date,
        periods=FORECAST_HORIZON,
        freq="M"
    )

forecast_df.index = future_index
forecast_df.index.name = "forecast_date"

print(f"✓ 예측 DataFrame 생성: {forecast_df.shape}")
print(f"✓ 예측 기간: {future_index[0]} ~ {future_index[-1]}")


# ==================== 9. 결과 표시 ====================
print("\n" + "=" * 70)
print("예측 결과 테이블")
print("=" * 70)
print(forecast_df.head(10))

print("\n" + "=" * 70)
print("기술통계")
print("=" * 70)
print(forecast_df.describe())


# ==================== 10. 저장 (선택사항) ====================
print("\n" + "=" * 70)
print("결과 저장")
print("=" * 70)

# # 저장 경로는 현재 위치 기준
# output_path = paths['current'] / f"forecast_{TARGET_TICKER}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
#
# try:
#     forecast_df.to_csv(output_path, encoding='utf-8-sig')
#     print(f"✓ 결과 저장 완료: {output_path}")
# except Exception as e:
#     print(f"⚠ 저장 실패: {e}")

print("\n" + "=" * 70)
print("✅ 모든 작업 완료!")
print("=" * 70)

📁 경로 설정 완료
✓ 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
✓ DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✓ 현재 위치:     C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Korea_Market\analysis
✓ 운영체제:      nt

✓ 예측 모듈 import 성공
✓ 모든 모듈 import 완료

⚙️  설정 정보
  - 대상 티커: A005930
  - 예측 기간: 12분기
  - 사용 모델: SARIMA, ETS, Prophet, LSTM
  - 데이터베이스: 192.168.0.230:3307

1단계: 데이터 로드
✅ 'korea_fs_data' 테이블에서 5902708건의 데이터를 가져왔습니다.
✓ 데이터 로드 완료: 5,902,708행

2단계: 데이터 전처리
✓ 지표 필터링: 순이익률 - 170,100행
✓ 피벗 테이블 생성: (87, 3254)
✓ 날짜 범위: 2010-03-31 00:00:00 ~ 2025-06-30 00:00:00
✓ 티커 수: 3,254개

메모리 상태 확인:

3단계: 예측 실행
[메모리] forecast_one_from_pivot_inline 실행 전: 1408.06 MB

[예측 중] SARIMA 모델...
[메모리] forecast_sarima 실행 전: 1408.06 MB
[메모리] find_best_sarima_params 실행 전: 1408.06 MB
[메모리] find_best_sarima_params 실행 후: 1409.20 MB (변화: +1.14 MB)
[메모리] forecast_sarima 실행 후: 1409.20 MB (변화: +1.14 MB)

[예측 중] ETS 모델...
[메모리] forecast_ets 실행 전: 1409.20 MB
[메모리

22:53:49 - cmdstanpy - INFO - Chain [1] start processing



[예측 중] Prophet 모델...
[메모리] forecast_prophet 실행 전: 1409.21 MB


22:53:49 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 1409.21 MB (변화: +0.00 MB)

[예측 중] LSTM 모델...
[메모리] forecast_lstm 실행 전: 1409.21 MB
[메모리] forecast_lstm 실행 후: 1436.22 MB (변화: +27.01 MB)
[메모리] forecast_one_from_pivot_inline 실행 후: 1436.22 MB (변화: +28.16 MB)

✓ 예측 완료!

4단계: 예측 결과

▶ SARIMA
  · 모델 사양: {'order': (2, 0, 1), 'seasonal_order': (0, 0, 0, 4), 'ic_value': -193.19242888496714}
  · 예측값 (처음 5개): [0.11 0.12 0.12 0.12 0.12]
  · 예측값 범위: 0.11 ~ 0.12

▶ ETS
  · 모델 사양: {'seasonal_periods': 4, 'trend': 'add', 'seasonal': 'add'}
  · 예측값 (처음 5개): [0.1  0.1  0.09 0.09 0.1 ]
  · 예측값 범위: 0.09 ~ 0.10

▶ Prophet
  · 모델 사양: {'seasonality_mode': 'multiplicative'}
  · 예측값 (처음 5개): [0.13 0.13 0.13 0.13 0.13]
  · 예측값 범위: 0.13 ~ 0.13

▶ LSTM
  · 모델 사양: {'lookback': 12, 'epochs': 50, 'batch_size': 16}
  · 예측값 (처음 5개): [0.11 0.11 0.1  0.1  0.11]
  · 예측값 범위: 0.10 ~ 0.11

5단계: 결과 DataFrame 생성
✓ 마지막 날짜: 2025-06-30 00:00:00
✓ 주기: Q
✓ 예측 DataFrame 생성: (12, 4)
✓ 예측 기간: 2025-09-30 00:00:00 ~ 2028-06-30 00:00:00

예측 결과 테이블
          